# ***Assignment 2E, 2026 - Black-box VI for a Poisson–Gamma model (Question 2.3.11)***

In Question 2.3.9 you derived the naive BBVI gradient for the model
$$
x_n\mid\lambda\overset{\text{iid}}{\sim}\mathrm{Poisson}(\lambda),\quad n=1,\dots,N,\qquad \lambda\sim\mathrm{Gamma}(\alpha_0,\beta_0)\ \text{(shape, rate)},
$$
with variational distribution $q(\lambda)=\mathrm{Exponential}(\theta)$. Here you implement it on real count data, add the control variate of Ranganath et al. (2014), and repeat the experiment with a richer family $q(\lambda)=\mathrm{Gamma}(a,b)$.

Because the model is conjugate, the exact posterior $\mathrm{Gamma}(\alpha_0+\sum_n x_n,\ \beta_0+N)$ and the ELBO of both variational families are available in closed form. You can therefore check whether your black-box estimates are correct, even though BBVI itself never uses these formulas.

Use **numpy and scipy only**, and do not change the seeds.

**Data.** Capital Bikeshare, Washington D.C., 2011 (Fanaee-T & Gama, 2013; copy from the ISLP package, file `data/Bikeshare.csv`). We model the number of *registered* users starting a trip between 4 and 5 a.m. on the first $N=30$ working days of the year. At this hour the counts are small and roughly Poisson-like (they are, if anything, slightly under-dispersed).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import gammaln, digamma
from scipy.stats import gamma as gamma_dist

SEED = 2026  # Do not change the seed

## Provided code

In [ ]:
bikes = pd.read_csv("1E/data/Bikeshare.csv")
counts = bikes.loc[(bikes["hr"] == 4) & (bikes["workingday"] == 1), "registered"].to_numpy()
x = counts[:30]
N, sum_x = len(x), x.sum()
print(f"N = {N}, counts: {x}")
print(f"mean = {x.mean():.2f}, variance = {x.var(ddof=1):.2f}")

ALPHA0, BETA0 = 2.0, 0.5    # Gamma prior (shape, rate): prior mean 4 riders per hour
A_POST, B_POST = ALPHA0 + sum_x, BETA0 + N
print(f"exact posterior: Gamma({A_POST:.0f}, {B_POST:.1f}), mean {A_POST / B_POST:.3f}")


def log_joint(lam, x=x):
    # log p(x, lambda) for an array of lambda values
    lam = np.asarray(lam, dtype=float)
    return (np.sum(x) * np.log(lam) - len(x) * lam - np.sum(gammaln(x + 1))
            + ALPHA0 * np.log(BETA0) - gammaln(ALPHA0) + (ALPHA0 - 1) * np.log(lam) - BETA0 * lam)


def elbo_gamma(a, b, x=x):
    # Closed-form ELBO for q(lambda) = Gamma(a, b) (rate b). Exponential(theta) is Gamma(1, theta).
    E_lam, E_log_lam = a / b, digamma(a) - np.log(b)
    entropy = a - np.log(b) + gammaln(a) + (1 - a) * digamma(a)
    return (np.sum(x) * E_log_lam - len(x) * E_lam - np.sum(gammaln(x + 1))
            + ALPHA0 * np.log(BETA0) - gammaln(ALPHA0) + (ALPHA0 - 1) * E_log_lam - BETA0 * E_lam + entropy)


def log_evidence(x=x):
    # Closed-form log p(x); KL(q || posterior) = log p(x) - ELBO
    a, b = ALPHA0 + np.sum(x), BETA0 + len(x)
    return (ALPHA0 * np.log(BETA0) - gammaln(ALPHA0) + gammaln(a) - a * np.log(b) - np.sum(gammaln(x + 1)))


def control_variate_scaling(f, h):
    # Leave-one-out estimate of a* = Cov(f, h) / Var(h) for each sample s, computed without sample s.
    # Estimating a* from the same samples it is applied to makes the estimator biased for small S;
    # leaving sample s out keeps E[f_s - a_{-s} h_s] = E[f] because a_{-s} is independent of z_s.
    S = len(f)
    n = S - 1
    mean_f = (f.sum() - f) / n
    mean_h = (h.sum() - h) / n
    cov = ((f * h).sum() - f * h - n * mean_f * mean_h) / (n - 1)
    var = ((h * h).sum() - h * h - n * mean_h**2) / (n - 1)
    return cov / var


def adagrad_ascent(grad_fn, omega0, n_iter, lr, rng):
    # Stochastic gradient ascent with AdaGrad step sizes on unconstrained parameters omega.
    omega, G = np.array(omega0, dtype=float), np.zeros(len(omega0))
    trace = np.zeros((n_iter, len(omega0)))
    for t in range(n_iter):
        g = grad_fn(omega, rng)
        G += g**2
        omega = omega + lr * g / (np.sqrt(G) + 1e-8)
        trace[t] = omega
    return trace

### Part (a): naive BBVI with $q(\lambda)=\mathrm{Exponential}(\theta)$

We optimise the unconstrained parameter $\omega=\log\theta$, so by the chain rule $\nabla_\omega\log q=\theta\,\nabla_\theta\log q$.

Implement `grad_exponential(omega, rng, S, control_variate=False)`, which returns the BBVI estimate of $\nabla_\omega\mathcal L$ from $S$ samples $z_s\sim q$ (Ranganath et al., Algorithm 1 for `control_variate=False`).

Then run `adagrad_ascent` for 3000 iterations with $S=10$, learning rate 0.5 and $\omega_0=0$. Plot $\theta_t$ together with the value $\theta^\star$ that maximises the closed-form ELBO. Find $\theta^\star$ numerically with `elbo_gamma(1, theta)` on a grid, or derive it.

Compare $\mathbb E_q[\lambda]=1/\theta^\star$ with the exact posterior mean, and compare the variance of $q$ with the posterior variance. What does this say about the Exponential family as a variational family here?

In [ ]:
def grad_exponential(omega, rng, S=10, control_variate=False):
    # Insert your code here
    return grad

In [ ]:
# Run BBVI, plot theta_t and theta*, and compare q with the exact posterior

*Your discussion here.*

### Part (b): control variates

Extend `grad_exponential` with the control variate of Ranganath et al. (2014, Section 3.2): $h(z)=\nabla_\omega\log q(z)$, which has expectation zero, and scaling $\hat a=\widehat{\mathrm{Cov}}(f,h)/\widehat{\mathrm{Var}}(h)$ estimated from the $S$ samples with the provided `control_variate_scaling`, which leaves each sample out of its own scaling so that the estimator stays unbiased (`control_variate=True`).

At $\theta\in\{0.5\,\theta^\star,\ \theta^\star,\ 2\,\theta^\star\}$, estimate the variance of the gradient estimator with and without the control variate from 2000 independent estimates with $S=10$, and report the results in a table. Rerun part (a) with the control variate and overlay the two traces. Explain why this control variate reduces the variance.

In [ ]:
# Insert your code here

*Your table and discussion here.*

### Part (c): a richer family, $q(\lambda)=\mathrm{Gamma}(a,b)$

Implement `grad_gamma(omega, rng, S, control_variate)` for the unconstrained parameters $\omega=(\log a,\log b)$, using the scores $\nabla_{\log a}\log q=a\,(\log b-\psi(a)+\log z)$ and $\nabla_{\log b}\log q=a-bz$. Use one control variate per component.

Run `adagrad_ascent` for 20000 iterations with $S=50$, learning rate 0.5 and $\omega_0=(0,0)$, with and without the control variate. For both runs:
- plot the ELBO (`elbo_gamma`) over iterations together with $\log p(x)$ (`log_evidence`);
- report the final $(a,b)$, averaged over the last 2000 iterations, and compare with the exact posterior $\mathrm{Gamma}(\alpha_0+\sum x_n,\ \beta_0+N)$;
- plot the two variational densities together with the exact posterior.

Why can BBVI with this family recover the exact posterior, and why does the naive estimator struggle?

In [ ]:
def grad_gamma(omega, rng, S=50, control_variate=True):
    # Insert your code here
    return grad

In [ ]:
# Insert your code here

*Your discussion here.*